# Статистичні гіпотези

`03_visualizations.ipynb` показав два стійкі візуальні патерни: час доставки впливає на оцінку клієнта, і середній чек відрізняється між способами оплати. Але візуальне спостереження — не доказ: різниця може бути випадковою або наслідком нерівних вибірок.

Цей ноутбук формально перевіряє обидва спостереження через статистичні тести — підтверджує або спростовує що знайдені патерни є реальними, а не шумом у даних.

| Гіпотеза | Тест | Питання |
|---|---|---|
| 1 | t-test (`scipy.stats.ttest_ind`) | Чи впливає час доставки на оцінку клієнта? |
| 2 | Kruskal-Wallis (`scipy.stats.kruskal`) | Чи відрізняється середній чек залежно від способу оплати? |

**Рівень значущості для обох тестів:** α = 0.05

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats

sys.path.append(str(Path.cwd().parent))
from src.config import get_engine

engine = get_engine()

print("✓ Підключення до БД встановлено")

✓ Підключення до БД встановлено


### Гіпотеза 1: Вплив часу доставки на оцінку клієнта

**Мета:** Перевірити, чи існує статистично значущий зв'язок між швидкістю 
доставки та задоволеністю клієнта (оцінка відгуку 1–5).

**Бізнес-контекст:** У `03_visualizations.ipynb` (Графік 5, Q8) виявлено, що середня оцінка 
помітно знижується при доставці понад ~15 днів. Однак візуальне спостереження 
не є доказом — різниця може бути випадковою. Статистичний тест дозволяє 
відповісти: чи є ця різниця реальною, чи просто шумом у даних?

**Поріг розділення груп:** 7 днів — медіана часу доставки по платформі,
що ділить вибірку на дві рівноцінні групи для порівняння.

**Гіпотези:**
- **H₀ (нульова):** Середня оцінка клієнтів з доставкою ≤ 7 днів 
  дорівнює середній оцінці клієнтів з доставкою > 7 днів
- **H₁ (альтернативна):** Середні оцінки між групами відрізняються

**Рівень значущості:** α = 0.05

**Вибір тесту:** t-test для двох незалежних вибірок (`scipy.stats.ttest_ind`) —
порівнює середні значення двох груп і визначає, чи є різниця між ними
статистично значущою, чи виникла випадково.

**Чому t-test, а не z-test:** z-test вимагає відомого генерального
стандартного відхилення (σ) популяції. Оскільки робота йде з вибіркою
і σ невідоме — оцінюється з даних, тому застосовується t-test.

**Чому t-test коректний без перевірки нормальності:** вибірка містить
десятки тисяч замовлень, тому згідно з центральною граничною теоремою
розподіл вибіркових середніх наближається до нормального —
t-test коректно застосовувати навіть без перевірки нормальності
вихідних даних.

In [2]:
sql_t1 = """
    SELECT 
        (o.delivered_to_customer_at::date - o.purchased_at::date) AS delivery_days,
        r.score
    FROM orders o
    JOIN order_reviews r ON o.order_id = r.order_id
    WHERE o.delivered_to_customer_at IS NOT NULL
"""

df_t1 = pd.read_sql(sql_t1, engine)

print(df_t1.shape)
display(df_t1.head())

(95607, 2)


,delivery_days,score
0,8,4
1,14,4
2,9,5
3,14,5
4,3,5


In [3]:
group_fast = df_t1[df_t1['delivery_days'] <= 7]['score']
group_slow = df_t1[df_t1['delivery_days'] > 7]['score']

print(f'Група ≤7 днів: {len(group_fast):,} замовлень, середня оцінка: {group_fast.mean():.3f}')
print(f'Група >7 днів: {len(group_slow):,} замовлень, середня оцінка: {group_slow.mean():.3f}')

Група ≤7 днів: 30,498 замовлень, середня оцінка: 4.413
Група >7 днів: 65,109 замовлень, середня оцінка: 4.038


In [4]:
t_stat, pval = stats.ttest_ind(group_fast, group_slow)

ci_fast = stats.t.interval(
                            confidence=0.95, 
                            df=len(group_fast)-1, 
                            loc=group_fast.mean(), 
                            scale=stats.sem(group_fast)
)
ci_slow = stats.t.interval(
                            confidence=0.95, 
                            df=len(group_slow)-1, 
                            loc=group_slow.mean(), 
                            scale=stats.sem(group_slow)
)

print(f't statistic: {t_stat:.2f}')
if pval < 0.001:
    print(f'p-value: < 0.001 (фактично ≈ 0)')
else:
    print(f'p-value: {pval:.3f}')
print(f'Довірчий інтервал 95% для групи ≤7 днів: [{ci_fast[0]:.3f}, {ci_fast[1]:.3f}]')
print(f'Довірчий інтервал 95% для групи >7 днів: [{ci_slow[0]:.3f}, {ci_slow[1]:.3f}]')

t statistic: 42.54
p-value: < 0.001 (фактично ≈ 0)
Довірчий інтервал 95% для групи ≤7 днів: [4.401, 4.425]
Довірчий інтервал 95% для групи >7 днів: [4.028, 4.048]


In [5]:
mean_diff = group_fast.mean() - group_slow.mean()
pooled_std = np.sqrt((group_fast.std()**2 + group_slow.std()**2) / 2)
cohens_d = mean_diff / pooled_std

if cohens_d < 0.2:
    effect = 'малий'
elif cohens_d < 0.5:
    effect = 'середній'
else:
    effect = 'великий'

print(f"Cohen's d: {cohens_d:.3f} ({effect} ефект)")

Cohen's d: 0.307 (середній ефект)


### Висновок

Різниця між оцінками користувачів для різних періодів доставки є статистично значущою. Оскільки p-value < 0.001 є значно меншим за рівень значущості (α = 0.05), відхиляється нульова гіпотеза: клієнти з доставкою ≤ 7 днів залишають статистично значущо вищі оцінки, ніж клієнти з доставкою > 7 днів (4.413 проти 4.038).

Довірчі інтервали груп не перетинаються ([4.401, 4.425] та [4.028, 4.048]), що візуально підтверджує різницю: з 95% впевненістю істинні середні оцінки кожної групи лежать у різних діапазонах.

Cohen's d = 0.307 вказує на **середній практичний ефект**. Це означає, що різниця не лише статистично значуща, але й має реальне практичне значення для бізнесу — швидкість доставки помітно впливає на задоволеність клієнтів.

**Бізнес-висновок:** t-test статистично підтверджує зв'язок між часом доставки і задоволеністю. Графік 5 (`03_visualizations.ipynb`, Q8) уточнює, де відбувається критичне падіння оцінки: при переході через 15 днів середня оцінка падає з 4.29 до 3.65 — найрізкіший стрибок у всьому розподілі. 
  
**Рекомендація:** встановити SLA для логістичних партнерів на рівні **14 днів максимум** — це дозволить утримати 73% замовлень у зоні оцінки 4.0+.

### Гіпотеза 2: Спосіб оплати впливає на середній чек

**Мета:** Перевірити, чи існує статистично значущий зв'язок між способами оплати та середнім чеком.

**Бізнес-контекст:** У `03_visualizations.ipynb` (Графік 6, Q7) виявлено, що найбільша кількість оплат відбувається через Credit Card і має вищий середній чек. Однак візуальне спостереження не є доказом — різниця може бути випадковою. Статистичний тест дозволяє 
відповісти: чи є ця різниця реальною, чи просто шумом у даних?

**Гіпотези:**
- **H₀ (нульова):** Середній чек однаковий для всіх трьох способів оплати
- **H₁ (альтернативна):** Принаймні один спосіб оплати має статистично відмінний середній чек

**Рівень значущості:** α = 0.05

**Вибір тесту:** Kruskal-Wallis (`scipy.stats.kruskal`) — непараметричний 
аналог ANOVA для порівняння трьох і більше незалежних груп. 
Не вимагає нормального розподілу, що важливо для сум чеків — 
розподіл яких є правосторонньо скошеним (skewed): багато малих 
покупок і кілька дуже великих.

**Чому не ANOVA та не Хі-квадрат:** ANOVA порівнює середні значення, 
але вимагає нормального розподілу — що не виконується для сум чеків. 
Хі-квадрат використовують для категоріальних даних (частоти, кількості) — в цій задачі порівнюються числові значення, тому він не підходить.

**Примітка:** Спосіб оплати `debit_card` виключено з аналізу через незначну частку транзакцій у датасеті — результати тесту були б ненадійними через малу вибірку.

In [6]:
sql_t2 = """
    SELECT 
        payment_type, 
        value
    FROM order_payments
    WHERE payment_type IN ('credit_card', 'boleto', 'voucher')
"""

df_t2 = pd.read_sql(sql_t2, engine)

print(df_t2.shape)
display(df_t2.head())

(102354, 2)


,payment_type,value
0,credit_card,99.33
1,credit_card,24.39
2,credit_card,65.71
3,credit_card,107.78
4,credit_card,128.45


In [7]:
group_cc = df_t2[df_t2['payment_type'] == 'credit_card']['value']
group_boleto = df_t2[df_t2['payment_type'] == 'boleto']['value']
group_voucher = df_t2[df_t2['payment_type'] == 'voucher']['value']

print(f'credit_card: {len(group_cc):,} платежів, середній чек: {group_cc.mean():.2f}')
print(f'boleto:      {len(group_boleto):,} платежів, середній чек: {group_boleto.mean():.2f}')
print(f'voucher:     {len(group_voucher):,} платежів, середній чек: {group_voucher.mean():.2f}')

credit_card: 76,795 платежів, середній чек: 163.32
boleto:      19,784 платежів, середній чек: 145.03
voucher:     5,775 платежів, середній чек: 65.70


In [8]:
stat, pval = stats.kruskal(group_cc, group_boleto, group_voucher)

print(f'Kruskal-Wallis statistic: {stat:.2f}')
if pval < 0.001:
    print(f'p-value: < 0.001 (фактично ≈ 0)')
else:
    print(f'p-value: {pval:.3f}')

Kruskal-Wallis statistic: 5458.62
p-value: < 0.001 (фактично ≈ 0)


### Висновок

Різниця між середнім чеком для різних способів оплати є статистично значущою. Оскільки p-value < 0.001 є значно меншим за рівень значущості (α = 0.05), відхиляється нульова гіпотеза: клієнти зі способом оплати Credit Card мають статистично значущо вищий середній чек, ніж клієнти з іншими засобами оплати (163.94 проти 145.03 та 98.15).

**Бізнес-висновок:**
Спосіб оплати `voucher` не генерує великий GMV — низький середній чек (65.70) є очікуваною поведінкою, адже клієнти використовують ваучери для знижок. Активно промоутити цей спосіб оплати платформі невигідно.

Оскільки `credit_card` дає найвищий середній чек (163.32), Olist може стимулювати його використання через cashback або розстрочку (installments) — це додатково збільшить GMV платформи.

`Boleto` охоплює аудиторію без доступу до кредитної картки і генерує порівнянний середній чек (145.03). Цей спосіб оплати варто зберігати як обов'язкову опцію, щоб не втратити цей сегмент аудиторії.

## Загальний висновок

Обидва статистичні тести відхилили нульові гіпотези (p < 0.001). Знайдені візуально патерни є реальними і статистично значущими:

**Гіпотеза 1 підтверджена.** Час доставки статистично значуще впливає на оцінку клієнта (Cohen's d = 0.307 — середній практичний ефект). Критичний поріг — 15 днів: саме там оцінка падає найрізкіше. Рекомендований SLA — 14 днів.

**Гіпотеза 2 підтверджена.** Спосіб оплати статистично значуще пов'язаний із середнім чеком. Credit Card генерує найвищий GMV — стимулювання його частки через cashback або розстрочку дасть вимірюваний ефект на виручку.

Зведені бізнес-висновки та рекомендації на основі всього аналізу — у `05_conclusions.ipynb`.